# LoRA Fine-Tuning: Qwen2.5-0.5B-Instruct
## Layer Ablation Study on MMLU & GSM8K

This notebook implements LoRA across three layer configurations:
- **All layers** — every transformer block (0–23)
- **Middle layers** — blocks 7–21
- **Late layers** — blocks 14–23 (> block 14)

Evaluated on MMLU (5-shot, multiple choice) and GSM8K (8-shot, math reasoning).

> **Runtime**: Use `Runtime → Change runtime type → T4 GPU` before running.

In [ ]:
# !pip install pdfminer
# !pip install pymupdf
# !pip install pdfplumber
# !pdf2txt.py ooo.pdf > ooo.txt

In [ ]:
# import pymupdf
# doc = pymupdf.open("ooo.pdf")
# text = "\n".join([page.get_text() for page in doc])
# import pdfplumber
# doc = pdfplumber.open("ooo.pdf")
# text = "\n".join([page.extract_text() for page in doc.pages])

# print(text)
# with open('ooo.txt', 'w') as f: f.write(text)

## 1. pip install

In [ ]:
%%capture
# transformers: 4.47.0, 5.8.0 (qwen3)
# !pip uninstall transformers
!pip install -q transformers==5.8.0 peft==0.12.0 trl==0.12.0 \
    sentencepiece protobuf datasets evaluate rouge_score \
    tokenizers==0.22.2 bitsandbytes==0.49.2 accelerate==1.13.0 \
    # torch==2.9.1

# !pip uninstall torch -y
# !pip install torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 --index-url https://download.pytorch.org/whl/cu128
# !pip install ninja # faster build
# !pip install flash-attn==2.8.3
# !nvidia-smi
# !nvcc --version # cuda version
# !python --version
# !pip show torch
# !pip freeze

In [ ]:
# !wget -P /content/drive/MyDrive https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
# !ls /content/drive/MyDrive

# !pip install ninja # faster build
# !pip install flash-attn==2.8.3

# !pip install /content/drive/MyDrive/flash_attn-2.8.3+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
# !python -c "from flash_attn import flash_attn_func; print('Flash Attention installed successfully')"

In [ ]:
# !ypip uninstall transformers
# !pip install transformers
# import transformers
# print(transformers.__version__)
!pip show transformers | grep Version
# print(torch.__version__)
# !pip install "lm_eval[hf]"
# !lm-eval ls tasks
# !lm_eval --model hf \
#     --model_args pretrained=EleutherAI/pythia-2.8b \
#     --tasks mmlu \
#     --device cuda:0 \
#     --num_fewshot 0
    # --model_args pretrained=Qwen/Qwen2.5_7B_Instruct \
    # --batch_size 8 \
# from google.colab import userdata
# userdata.get('HF_TOKEN')

# from huggingface_hub import login
# login(token=os.environ.get("HF_TOKEN"))

# from huggingface_hub import login
# from google.colab import userdata
# HF_TOKEN=userdata.get('HF_TOKEN')
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# if HF_TOKEN:
#     login(HF_TOKEN)
#     print("Successfully logged in to Hugging Face!")
# else:
#     print("Token is not set. Please save the token first.")

# from huggingface_hub import whoami
# print(whoami())
# from huggingface_hub import model_info
# print(model_info("meta-llama/Llama-3.2-1B-Instruct", token=HF_TOKEN))

# MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
# MODEL_ID = 'google/gemma-3-270m'
# MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID) #, token=HF_TOKEN)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     # torch_dtype="auto",
#     torch_dtype=torch.float16,
#     device_map="auto",
#     attn_implementation="flash_attention_2",
#     trust_remote_code=True,
#     # token=HF_TOKEN,
# )

# %rm -rf /content/drive/MyDrive/LoRA/Data


Version: 5.8.0


## 2. import, mount

In [ ]:
import gc, os, re, json, time, random, warnings
import numpy as np
import torch
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from trl import DataCollatorForCompletionOnlyLM
from google.colab import runtime

from transformers import (
    AutoConfig, AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig,
    # DataCollatorForLanguageModeling,
    set_seed,
    TrainingArguments, Trainer,
    Gemma3ForConditionalGeneration, Gemma3ForCausalLM,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings('ignore')
set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'torch: {torch.__version__}')

# OUTPUT_DIR = '/content/lora_outputs'
# os.makedirs(OUTPUT_DIR, exist_ok=True)

## mount google drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
OUTPUT_DIR = '/content/drive/MyDrive/LoRA/Data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# os.environ['HF_DATASETS_CACHE'] = '/content/drive/MyDrive/hf_cache/'
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache/'

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
torch: 2.11.0+cu128
Mounted at /content/drive


## 3. config

In [ ]:
# models = [
#     'Qwen/Qwen2.5-0.5B',
#     # 'Qwen/Qwen2.5-0.5B-Instruct',
#     # 'Qwen/Qwen2.5-1.5B-Instruct',
#     # 'Qwen/Qwen2.5-3B-Instruct',
#     # 'Qwen/Qwen2.5-7B-Instruct',
#     # 'Qwen/Qwen2.5-14B-Instruct',
#     # 'Qwen/Qwen3-0.6B',
#     # 'Qwen/Qwen3-1.7B',
#     # 'Qwen/Qwen3-4B',
#     # 'Qwen/Qwen3-4B_Instruct_2507',
#     # 'Qwen/Qwen3-8B',
#     # 'Qwen/Qwen3-14B',
#     # 'meta-llama/Llama-3.2-1B-Instruct',
#     # 'meta-llama/Llama-3.2-3B-Instruct',
#     # 'meta-llama/Llama-3.1-8B-Instruct',
# ]

## LoRA layer config
def lora_layers(a, c=0):
    b = a // 3
    all = list(range(0, a))
    early = list(range(0, b+c))
    mid = list(range(b-c, 2*b+c))
    late = list(range(2*b-c, a))
    rand = random.sample(all, len(all) // 3)
    return all, rand, early, mid, late

# ─── Global Config ────────────────────────────────────────────────────────────
# MODEL_ID = models[0]
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
# MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
# MODEL_ID = 'google/gemma-2-2b-it'

config = AutoConfig.from_pretrained(MODEL_ID)

TOTAL_LAYERS = config.num_hidden_layers   # Qwen2.5-0.5B has 24 transformer layers (0-indexed: 0–23)

# Layer ranges — adjust boundaries here if you want to experiment
# LAYER_CONFIGS = {
    # 'all':    list(range(0, 24)),          # all 24 blocks
    # 'middle': list(range(7, 22)),          # blocks 7–21 (inclusive)
    # 'late':   list(range(14, 24)),         # blocks > 14 → 15–23 + include 14
# }
# NOTE: The paper spec says ">14" so blocks 15-23; adjust below if you want >=14
# LAYER_CONFIGS['late'] = list(range(15, 24))  # strict > 14

all, rand, early, mid, late = lora_layers(TOTAL_LAYERS, 0)

LAYER_CONFIGS = {'all': all, 'rand': rand, 'early': early, 'mid': mid, 'late': late}
# LAYER_CONFIGS = {'late': late}

# LoRA hyper-parameters
LORA_R          = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                   'gate_proj', 'up_proj', 'down_proj']

# Training hyper-parameters
BATCH_SIZE          = 4
GRAD_ACCUM          = 4          # effective batch = 16; BS=2 GA=8 halves activation mem
TRAIN_EPOCHS        = 1          # 1 increase to 2-3 for better convergence
LR                  = 5e-5 # 2e-4 5e-5
MAX_SEQ_LEN         = 1024 # 512 qwen 1024, gemma-2b 512
WARMUP_RATIO        = 0.05
TRAIN_SAMPLES       = 10000       # per dataset; reduce if OOM, 2000 to 5000, 1816 mmlu
# EVAL_SAMPLES        = 200        # quick eval; set to None for full benchmark 200

print(f'MODEL_ID: {MODEL_ID}')
print(f'nLayers:{TOTAL_LAYERS}')
print('layers config:')
for name, layers in LAYER_CONFIGS.items():
    print(f'  {name:8s}: {len(layers)} layers — blocks {layers[0]}–{layers[-1]}')



MODEL_ID: Qwen/Qwen2.5-0.5B-Instruct
nLayers:24
layers config:
  all     : 24 layers — blocks 0–23
  rand    : 8 layers — blocks 20–22
  early   : 8 layers — blocks 0–7
  mid     : 8 layers — blocks 8–15
  late    : 8 layers — blocks 16–23


In [ ]:
LAYER_CONFIGS

{'all': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25],
 'rand': [20, 3, 0, 8, 7, 21, 4, 24],
 'early': [0, 1, 2, 3, 4, 5, 6, 7],
 'mid': [8, 9, 10, 11, 12, 13, 14, 15],
 'late': [16, 17, 18, 19, 20, 21, 22, 23, 24, 25]}

## 4. model / LoRA

In [ ]:
def cleanup_cuda():

    gc.collect() # initial gc

    # 1 explicitly free large CUDA tensors from Python heap
    freed = 0
    for obj in gc.get_objects():
        try:
            if isinstance(obj, torch.Tensor) and obj.is_cuda and obj.numel() > 1e6:
                freed += obj.numel() * obj.element_size()
                obj.data = torch.empty(0, device='cpu')  # replace data with empty CPU tensor
        except (ReferenceError, RuntimeError):
            pass

    # 2 find and delete lingering tensors/models in global scope
    for name in list(globals().keys()):
        obj = globals()[name]
        if isinstance(obj, torch.Tensor) and obj.device.type == 'cuda':
            print(f'  Lingering tensor: {name} — {obj.shape} {obj.dtype}')
            del globals()[name]
        elif isinstance(obj, torch.nn.Module):
            try:
                device = next(obj.parameters()).device.type
                allocated = sum(p.numel() * p.element_size() for p in obj.parameters()) / 1e9
                if device == 'cuda':
                    print(f'  Lingering module: {name} — {type(obj).__name__} ~{allocated:.2f} GB')
                    del globals()[name]
            except StopIteration:
                pass

    # 3 gc + cuda cache clear
    gc.collect()
    gc.collect() # 2ns pass for circular refs

    with torch.no_grad():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

    # 4 report
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  Freed ~{freed/1e9:.2f} GB | Allocated: {allocated:.2f} / {total:.2f} GB | Reserved: {reserved:.2f} GB')


def load_tokenizer(model_id):
    global tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    # tokenizer.padding_side = 'right'
    # tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def load_base_model(model_id):
    """Load in 4-bit (QLoRA-style) for memory efficiency."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, # 4bit, 8bit
        bnb_4bit_quant_type='nf4',  # nf4, fp4
        bnb_4bit_compute_dtype=torch.bfloat16,
        # bnb_4bit_use_double_quant=True, # only at RAM limit
    )
    model = AutoModelForCausalLM.from_pretrained(
    # model = Gemma3ForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        torch_dtype=torch.float16, # T4 native
        # torch_dtype=torch.bfloat16, # gemma3 4b 270m
        # attn_implementation="flash_attention_2",
    )
    model.config.use_cache = False
    model.enable_input_require_grads()
    return model

def make_lora_target_modules(layer_indices: List[int]) -> List[str]:
    """
    Build PEFT target_modules list restricted to specific layer indices.
    Qwen2.5 internal layer names follow: model.layers.<idx>.<submodule>
    """
    modules = []
    for idx in layer_indices:
        for proj in TARGET_MODULES:
            if proj in ['q_proj', 'k_proj', 'v_proj', 'o_proj']:
                modules.append(f'model.layers.{idx}.self_attn.{proj}')
            else:
                modules.append(f'model.layers.{idx}.mlp.{proj}')
    return modules


def build_lora_model(base_model, layer_indices: List[int], run_name: str):
    """Apply LoRA adapter for the given layer subset."""
    target_modules = make_lora_target_modules(layer_indices)
    lora_cfg = LoraConfig(
        task_type       = TaskType.CAUSAL_LM,
        r               = LORA_R,
        lora_alpha      = LORA_ALPHA,
        lora_dropout    = LORA_DROPOUT,
        target_modules  = target_modules,
        bias            = 'none',
        inference_mode  = False,
    )
    model = get_peft_model(base_model, lora_cfg)
    trainable, total = model.get_nb_trainable_parameters()
    print(f'[{run_name}] Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
    return model


print('Model factory ready.')

Model factory ready.


## 5. eval

In [ ]:
# def get_assistant_prefix(tokenizer) -> str:
#     """Return the assistant turn opener for the current model's chat template."""
#     probe = tokenizer.apply_chat_template(
#         [{'role': 'user',      'content': 'X'},
#          {'role': 'assistant', 'content': 'Y'}],
#         tokenize=False, add_generation_prompt=False
#     )
#     # find what appears right before 'Y' in the rendered template
#     idx = probe.rfind('Y')
#     prefix = probe[:idx]
#     # extract just the last line/token that opens the assistant turn
#     # return prefix[prefix.rfind('\n') + 1:] if '\n' in prefix else prefix
#     for marker in ['<|', '<']:
#         last = prefix.rfind(marker)
#         if last != -1:
#             return prefix[last:]
#     return prefix

def get_assistant_prefix(tokenizer) -> str:
    """Return the assistant turn opener for the current model's chat template."""
    _USER = '⚙USR⚙'
    _ASST = '⚙AST⚙'
    probe = tokenizer.apply_chat_template(
        [{'role': 'user',      'content': _USER},
         {'role': 'assistant', 'content': _ASST}],
        tokenize=False, add_generation_prompt=False
    )
    start = probe.rfind(_USER) + len(_USER)
    end   = probe.rfind(_ASST)
    return probe[start:end]

@torch.no_grad()
def evaluate_mmlu(model, tokenizer, dataset, batch_size=8) -> Dict:
    model.eval()
    tokenizer.padding_side = 'right'
    choice_ids = [tokenizer.encode(' ' + c, add_special_tokens=False)[-1] for c in CHOICES]
    assistant_prefix = get_assistant_prefix(tokenizer)
    print(f'    Assistant prefix: {repr(assistant_prefix)}')
    print(f'    Choice ids: {list(zip(CHOICES, choice_ids))}')

    sample = dataset[0]
    prompt = sample['text'].split(assistant_prefix)[0] + assistant_prefix + 'Answer:'
    # print(repr(prompt[-150:]))
    print(f'    Occurrences of assistant_prefix: {sample["text"].count(assistant_prefix)}')

    correct, total = 0, 0
    examples = list(dataset)
    print(f'    MMLU examples: {len(examples)}')
    print(f'    Batch size: {batch_size}')
    print(f'    Tokenizer vocab size: {tokenizer.vocab_size}')
    print(f'    Tokenizer padding_side: {tokenizer.padding_side}')

    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        prompts, golds = [], []

        ## acc 0.42
        for ex in batch:
            prompt_part = ex['text'].split(assistant_prefix)[0] + assistant_prefix + 'Answer:'
            prompts.append(prompt_part)
            golds.append(ex['answer_letter'])

        # ## acc 0.40
        # for ex in batch:
        #     # for eval: rebuild prompt without answer (model-agnostic)
        #     msgs = [{'role': 'user', 'content': ex['prompt'].rstrip('Answer:').rstrip()}]
        #     prompt = tokenizer.apply_chat_template(
        #         msgs, tokenize=False, add_generation_prompt=True
        #     )
        #     prompts.append(prompt)
        #     golds.append(ex['answer_letter'])

        # ### debug acc 0.40
        # for ex in batch:
        #     user_content = ex['prompt']
        #     if user_content.endswith('Answer:'):
        #         user_content = user_content[:-len('Answer:')].rstrip()
        #     msgs = [{'role': 'user', 'content': user_content}]
        #     prompt = tokenizer.apply_chat_template(
        #         msgs, tokenize=False, add_generation_prompt=True
        #     ) + 'Answer:'
        #     prompts.append(prompt)
        #     golds.append(ex['answer_letter'])

        enc = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN
        ).to(DEVICE)

        out = model(**enc)
        attention_mask = enc['attention_mask']
        last_token_pos = attention_mask.sum(dim=1) - 1
        last_logits    = out.logits[
            torch.arange(out.logits.shape[0], device=out.logits.device),
            last_token_pos
        ]
        choice_logits = last_logits[:, choice_ids]
        preds = [CHOICES[j] for j in choice_logits.argmax(dim=-1).tolist()]

        ### debug
        if i == 0:
            # print(f'    attention_mask sum per seq: {enc["attention_mask"].sum(dim=1).tolist()}')
            # print(f'    last_token_pos: {last_token_pos.tolist()}')
            # print(f'    seq lengths in batch: {[len(tokenizer.encode(p)) for p in prompts]}')
            # last_pos = last_token_pos[0].item()
            # last_tok = enc['input_ids'][0][last_pos].item()
            # print(f'    Last token id: {last_tok} → {repr(tokenizer.decode([last_tok]))}')
            # last_5_ids = enc['input_ids'][0][-5:].tolist()
            # last_5_toks = [tokenizer.decode([t]) for t in last_5_ids]
            # print(f'    Last 5 tokens: {list(zip(last_5_ids, last_5_toks))}')
            # for k in range(len(preds)):
            #     print(f'    [{k}] Gold: {golds[k]} | Pred: {preds[k]} | '
            #           f'Choice logits: {list(zip(CHOICES, [round(x,2) for x in choice_logits[k].tolist()]))}')
            print(f'    NaNs in logits: {torch.isnan(out.logits).sum().item()}')
            print(f'    Logits min/max: {out.logits.min().item():.2f} / {out.logits.max().item():.2f}')

        for pred, gold in zip(preds, golds):
            correct += int(pred == gold)
            total   += 1

        if total % 50 == 0:
            print(f'    MMLU progress: {total}/{len(examples)} — running acc: {correct/total:.4f}')

    acc = correct / total if total > 0 else 0.0
    return {'mmlu_accuracy': acc, 'mmlu_correct': correct, 'mmlu_total': total}


def normalize_num_str(value):
    if value is None:
        return None
    value = value.replace(',', '').strip()
    try:
        number = float(value)
    except ValueError:
        return None
    if number.is_integer():
        return str(int(number))
    return str(number)


@torch.no_grad()
def evaluate_gsm8k(model, tokenizer, dataset, max_new_tokens=512, batch_size=16) -> Dict:
    """GSM8K evaluation — batched generation, extract numeric answer and compare."""

    model.eval()
    tokenizer.padding_side = 'left'

    assistant_prefix = get_assistant_prefix(tokenizer)
    print(f'    Assistant prefix: {repr(assistant_prefix)}')

    # tokenizer.padding_side = 'left'

    correct, total, empty_preds = 0, 0, 0
    examples = list(dataset)
    print(f'    GSM8K examples: {len(examples)}')
    print(f'    Batch size: {batch_size}')
    print(f'    Max new tokens: {max_new_tokens}')
    print(f'    Tokenizer vocab size: {tokenizer.vocab_size}')
    print(f'    Tokenizer padding_side: {tokenizer.padding_side}')

    if 'text' not in examples[0]:
        raise ValueError(f"Dataset missing 'text' column. Found: {list(examples[0].keys())}. "
                         f"Apply gsm8k_apply_chat_template() before passing to evaluate_gsm8k.")

    for i in range(0, len(examples), batch_size):
        t0 = time.time()
        batch = examples[i:i + batch_size]

        prompts, golds = [], []
        for ex in batch:
            # prompt = ex['text'].split('<|im_start|>assistant')[0] + '<|im_start|>assistant\n'
            prompt = ex['text'].split(assistant_prefix)[0] + assistant_prefix + '\n'
            prompts.append(prompt)
            golds.append(ex.get('numeric_answer', '').strip())

        enc = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(DEVICE)

        input_len = enc['input_ids'].shape[1]

        if i == 0:
            if DEVICE == 'cuda':
                torch.cuda.reset_peak_memory_stats()

            # logit NaN diagnostic — forward pass only, no generation
            with torch.no_grad():
                # diag_out = model(**enc)
                diag_enc = {k: v[:1] for k, v in enc.items()}  # first example only
                diag_out = model(**diag_enc)
            print(f'    NaNs in logits: {torch.isnan(diag_out.logits).sum().item()}')
            print(f'    Logits min/max: {diag_out.logits.min().item():.2f} / {diag_out.logits.max().item():.2f}')
            del diag_out
            torch.cuda.empty_cache()

        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )


        if i == 0:
            elapsed = time.time() - t0
            print(f'    Input length: {input_len} tokens')
            print(f'    Generated tokens: {out.shape[1] - input_len}')
            print(f'    First batch took {elapsed:.1f}s — estimated total: {elapsed * len(examples) / batch_size / 60:.1f} min')
            if DEVICE == 'cuda':
                total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
                print(f'    VRAM peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB / {total_vram:.2f} GB')


        for j, gold in enumerate(golds):
            generated = tokenizer.decode(out[j][input_len:], skip_special_tokens=False)
            generated = generated.split('<|im_end|>')[0].strip()
            generated = generated.replace('\\\\', '\\')

            # priority 1: \boxed{} format
            boxes = re.findall(r'\\boxed\{([-+]?\d[\d,]*(?:\.\d+)?)\}', generated)
            if boxes:
                pred = normalize_num_str(boxes[-1])
            else:
                # priority 2: last number in generation
                all_numbers = re.findall(r'[-+]?\d[\d,]*(?:\.\d+)?', generated)
                pred = normalize_num_str(all_numbers[-1]) if all_numbers else None

            gold_norm = normalize_num_str(gold)

            if pred is None:
                empty_preds += 1

            if total < 10 and pred != gold_norm:
                print(f'    Wrong [{total+1}] — Gold: {gold_norm} | Pred: {pred} | End: {repr(generated[-120:])}')

            correct += int(pred is not None and pred == gold_norm)
            total   += 1

        if total % 40 == 0:
            print(f'    GSM8K progress: {total}/{len(examples)} — running acc: {correct/total:.4f}')

    print(f'    Empty predictions: {empty_preds}/{total}')

    # tokenizer.padding_side = 'right'

    acc = correct / total if total > 0 else 0.0
    return {'gsm8k_accuracy': acc, 'gsm8k_correct': correct, 'gsm8k_total': total}


print('Evaluation utilities ready.')

Evaluation utilities ready.


## 6. train

In [ ]:
def run_experiment(config_name: str, layer_indices: List[int]) -> Dict:
    """
    Full experiment for one layer configuration:
      1. Load base model
      2. Apply LoRA to specified layers
      3. Fine-tune on combined MMLU+GSM8K
      4. Evaluate on both benchmarks
      5. Save adapter
    Returns dict with all metrics.
    """
    print(f'\n{"="*60}')
    print(f'EXPERIMENT: {config_name.upper()}  ({len(layer_indices)} layers)')
    print(f'{"="*60}')

    out_dir = os.path.join(OUTPUT_DIR, config_name)
    os.makedirs(out_dir, exist_ok=True)

    # --- Load base & wrap with LoRA ---
    print('\n[1/4] Loading base model ...')
    base = load_base_model(MODEL_ID)
    tokenizer = load_tokenizer(MODEL_ID)

    response_template_ids = tokenizer.encode(get_assistant_prefix(tokenizer), add_special_tokens=False)
    print(f'Response template: {repr(get_assistant_prefix(tokenizer))}')
    print(f'Response template ids: {response_template_ids}')

    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template_ids,
        tokenizer=tokenizer,
    )

    print('[2/4] Applying LoRA ...')
    model = build_lora_model(base, layer_indices, config_name)

    # --- Train ---
    print('[3/4] Fine-tuning ...')
    sft_cfg = SFTConfig(
        output_dir                  = out_dir,
        num_train_epochs            = TRAIN_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LR,
        lr_scheduler_type           = 'cosine',
        warmup_ratio                = WARMUP_RATIO,
        logging_steps               = 20,
        save_strategy               = 'epoch',
        fp16                        = True,
        # bf16                        = True,
        optim                       = 'paged_adamw_8bit',
        max_seq_length              = MAX_SEQ_LEN,
        dataset_text_field          = 'text', # 'text', 'prompt'
        packing                     = False,     # disable packing — it breaks loss masking
        report_to                   = 'none',
        seed                        = 42,
        # completion_only_loss        = True,   # replaces DataCollatorForCompletionOnlyLM, trl>=0.20
        # gradient_checkpointing = True, # reduce vram usage 60%, train 20% slower
        # gradient_checkpointing_kwargs = {'use_reentrant': False},
    )

    trainer = SFTTrainer(
        model           = model,
        tokenizer       = tokenizer,
        args            = sft_cfg,
        # train_dataset   = mmlu_train_sft, # mmlu_train_base
        train_dataset   = combined_train,
        data_collator   = collator,
    )

    t0 = time.time()
    train_result = trainer.train()
    time_train = time.time() - t0
    print(f'Train time: {time_train:.2f}s')

    vram_train = torch.cuda.max_memory_allocated() / 1e9
    print(f'Peak VRAM train: {vram_train:.2f} GB')

    # Save adapter weights
    model.save_pretrained(os.path.join(out_dir, 'adapter'))
    tokenizer.save_pretrained(os.path.join(out_dir, 'adapter'))
    print(f'  Adapter saved to {out_dir}/adapter')

    # --- Evaluate ---
    print('[4/4] Evaluating ...')
    model.eval()
    if hasattr(model, 'gradient_checkpointing_disable'):
        model.gradient_checkpointing_disable()

    ## debug # fingerprint using LoRA weights specifically
    lora_keys = [k for k in model.state_dict().keys() if 'lora_A' in k]
    print(f'LoRA keys count: {len(lora_keys)}')
    print(f'First LoRA key: {lora_keys[0]}')
    print(f'First LoRA sum: {model.state_dict()[lora_keys[0]].sum().item():.6f}')
    print(f'Last LoRA key: {lora_keys[-1]}')
    print(f'Last LoRA sum: {model.state_dict()[lora_keys[-1]].sum().item():.6f}')

    # verify adapter is actually being applied
    print(f'Model training mode: {model.training}')
    print(f'Active adapters: {model.active_adapters}')

    # sample_input = tokenizer('test', return_tensors='pt').to(DEVICE)
    # with torch.no_grad():
    #     out_with = model(**sample_input).logits[0, -1, :5]
    #     model.disable_adapters()
    #     out_without = model(**sample_input).logits[0, -1, :5]
    #     model.enable_adapters()

    # print(f'With adapter:    {[round(x,4) for x in out_with.tolist()]}')
    # print(f'Without adapter: {[round(x,4) for x in out_without.tolist()]}')
    # print(f'Adapters active: {not torch.allclose(out_with, out_without)}')


    t0 = time.time()
    torch.cuda.reset_peak_memory_stats()

    # Evaluate on MMLU
    mmlu_metrics = evaluate_mmlu(model, tokenizer, mmlu_test)
    print(f'  MMLU  accuracy: {mmlu_metrics["mmlu_accuracy"]:.4f} '
          f'({mmlu_metrics["mmlu_correct"]}/{mmlu_metrics["mmlu_total"]})')

    print(f'GSM8K test passed to evaluator: {len(gsm8k_test)} examples')
    print(f'First example keys: {list(gsm8k_test[0].keys())}')
    print(f'numeric_answer sample: {repr(gsm8k_test[0].get("numeric_answer", "MISSING"))}')

    # Evaluate on GSM8K (slower — generates text)
    # gsm8k_metrics = evaluate_gsm8k(model, tokenizer, gsm8k_test)
    gsm8k_metrics = evaluate_gsm8k(model, tokenizer, gsm8k_test, batch_size=32) # gsm8k_test bs=64 32, 14b: 16
    print(f'  GSM8K accuracy: {gsm8k_metrics["gsm8k_accuracy"]:.4f} '
          f'({gsm8k_metrics["gsm8k_correct"]}/{gsm8k_metrics["gsm8k_total"]})')

    time_eval = time.time() - t0
    print(f'Eval time: {time_eval:.2f}s')

    vram_eval = torch.cuda.max_memory_allocated() / 1e9
    print(f'Peak VRAM eval: {vram_eval:.2f} GB')

    # Compile results
    result = {
        'config':           config_name,
        'n_layers':         len(layer_indices),
        'layer_range':      f'{layer_indices[0]}–{layer_indices[-1]}',
        'train_loss':       train_result.training_loss,
        # 'train_time_s':     round(train_time, 1),
        'trainable_params': trainer.model.get_nb_trainable_parameters()[0],
        **mmlu_metrics,
        **gsm8k_metrics,
        'vram_train': vram_train,
        'time_train': time_train,
        'vram_eval': vram_eval,
        'time_eval': time_eval,
    }

    # Save per-experiment JSON
    with open(os.path.join(out_dir, 'results.json'), 'w') as f:
        json.dump(result, f, indent=2)

    # Free GPU memory before next run
    del model, base, trainer
    cleanup_cuda()

    return result


# response_template_ids = tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)
# print(f'Response template ids: {response_template_ids}')

## not needed without collator
print('Training loop defined.')

Training loop defined.


## 3. load data

In [ ]:
# ─── MMLU ─────────────────────────────────────────────────────────────────────
# We use the auxiliary_train split (57 subjects merged) for SFT,
# and 'test' split for evaluation.

# ## 13 subjects
# MMLU_SUBJECTS = [
#     'abstract_algebra', 'anatomy', 'astronomy', 'business_ethics',
#     'clinical_knowledge', 'college_biology', 'college_chemistry',
#     'college_computer_science', 'college_mathematics', 'college_medicine',
#     'high_school_mathematics', 'high_school_physics', 'high_school_chemistry'
# ]

## Use all 57 MMLU subjects instead of a hand-picked 13
MMLU_SUBJECTS = [
    'abstract_algebra', 'anatomy', 'astronomy', 'business_ethics',
    'clinical_knowledge', 'college_biology', 'college_chemistry',
    'college_computer_science', 'college_mathematics', 'college_medicine',
    'college_physics', 'computer_security', 'conceptual_physics',
    'econometrics', 'electrical_engineering', 'elementary_mathematics',
    'formal_logic', 'global_facts', 'high_school_biology',
    'high_school_chemistry', 'high_school_computer_science',
    'high_school_european_history', 'high_school_geography',
    'high_school_government_and_politics', 'high_school_macroeconomics',
    'high_school_mathematics', 'high_school_microeconomics',
    'high_school_physics', 'high_school_psychology', 'high_school_statistics',
    'high_school_us_history', 'high_school_world_history', 'human_aging',
    'human_sexuality', 'international_law', 'jurisprudence',
    'logical_fallacies', 'machine_learning', 'management', 'marketing',
    'medical_genetics', 'miscellaneous', 'moral_disputes', 'moral_scenarios',
    'nutrition', 'philosophy', 'prehistory', 'professional_accounting',
    'professional_law', 'professional_medicine', 'professional_psychology',
    'public_relations', 'security_studies', 'sociology', 'us_foreign_policy',
    'virology', 'world_religions'
]

CHOICES = ['A', 'B', 'C', 'D']

# def format_mmlu(example):
#     opts   = '\n'.join([f'{CHOICES[i]}. {example["choices"][i]}' for i in range(4)])
#     question = example['question'].strip()
#     answer   = CHOICES[example['answer']]
#     prompt   = (
#         f'Answer the following multiple-choice question. '
#         f'Respond with only the letter of the correct answer.\n\n'
#         f'Question: {question}\n{opts}\n\nAnswer:'
#     )
#     messages = [
#         {'role': 'user',      'content': prompt},
#         {'role': 'assistant', 'content': answer}
#     ]
#     text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
#     return {'text': text, 'answer_letter': answer, 'prompt': prompt}

def format_mmlu_raw(example):
    opts = '\n'.join([f'{CHOICES[i]}. {example["choices"][i]}' for i in range(4)])
    prompt = (
        f'Answer the following multiple-choice question. '
        f'Respond with only the letter of the correct answer.\n\n'
        f'Question: {example["question"].strip()}\n{opts}\n\nAnswer:'
    )
    return {'answer_letter': CHOICES[example['answer']], 'prompt': prompt}

def apply_chat_template(example):
    messages = [
        {'role': 'user',      'content': example['prompt']},
        {'role': 'assistant', 'content': example['answer_letter']}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

def mmlu_apply_chat_template(_base):
    return _base.map(apply_chat_template, num_proc=1, load_from_cache_file=False)


# ─── MMLU ────────────────────────────────────────────────────────────────────
def load_mmlu_raw(subjects, test_samples_per_subject=8, seed=42):
    print('Loading MMLU ...')
    train_parts, test_parts = [], []

    for subj in subjects:
        ds = load_dataset('cais/mmlu', subj)

        # train: merge auxiliary_train + validation + dev
        splits = [ds[s] for s in ['auxiliary_train', 'validation', 'dev'] if s in ds]
        if splits:
            train_parts.append(concatenate_datasets(splits))

        # test: balanced per subject
        test_split = ds['test'].shuffle(seed=seed)
        test_parts.append(test_split.select(range(min(test_samples_per_subject, len(test_split)))))

    train = concatenate_datasets(train_parts).shuffle(seed=seed)
    test  = concatenate_datasets(test_parts).shuffle(seed=seed)


    print(f'  MMLU — train: {len(train):,} | test: {len(test):,}')
    print(f'  train subjects: {len(set(train["subject"]))} | test subjects: {len(set(test["subject"]))}')
    return train, test


In [ ]:
# ─── Load once, never overwritten ─────────────────────────────────────────────
mmlu_train_raw, mmlu_test_raw = load_mmlu_raw(MMLU_SUBJECTS)

# clip only when needed, explicitly, in notebook scope
mmlu_train = mmlu_train_raw.select(range(min(TRAIN_SAMPLES, len(mmlu_train_raw))))

# model-agnostic formatting — run once
mmlu_train_base = mmlu_train.map(format_mmlu_raw, remove_columns=mmlu_train_raw.column_names, num_proc=4)
mmlu_test_base  = mmlu_test_raw.map(format_mmlu_raw,  remove_columns=mmlu_test_raw.column_names,  num_proc=4)
print(f'  MMLU raw — train: {len(mmlu_train_base):,} | test: {len(mmlu_test_base):,}')


Loading MMLU ...


README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

abstract_algebra/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 9.96kB            

abstract_algebra/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

  MMLU — train: 1,816 | test: 456
  train subjects: 57 | test subjects: 57


Map (num_proc=4):   0%|          | 0/1816 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/456 [00:00<?, ? examples/s]

  MMLU raw — train: 1,816 | test: 456


In [ ]:
# ─── GSM8K ────────────────────────────────────────────────────────────────────

GSM8K_SYSTEM = (
    'You are a math tutor. Solve the problem step by step, '
    'then give the final numeric answer on a new line in the format: #### <number>'
)

# GSM8K_SYSTEM = (
#     'You are a math tutor. Solve the problem step by step, '
#     'then give the final numeric answer on a new line in the format: #### <number>\n\n'
#     'Example:\nQuestion: Janet has 3 apples. She buys 5 more. How many does she have?\n'
#     'Step 1: 3 + 5 = 8\n'
#     '#### 8\n\n'
#     'Example:\nQuestion: A store has 20 shirts. They sell 7. How many are left?\n'
#     'Step 1: 20 - 7 = 13\n'
#     '#### 13\n\n'
# )

# GSM8K_SYSTEM = (
#     'You are a math tutor. Solve the problem step by step, '
#     'then give the final numeric answer on a new line in the format: #### <number>\n\n'
#     'Example:\n'
#     'Question: Janet has 3 apples. She buys 5 more. How many does she have?\n'
#     'She starts with 3 apples and buys 5 more: 3 + 5 = 8\n'
#     '#### 8\n\n'
#     'Example:\n'
#     'Question: A store has 20 shirts. They sell 7. How many are left?\n'
#     '20 - 7 = 13\n'
#     '#### 13\n\n'
#     'Example:\n'
#     'Question: Tom earns $12/hour and works 8 hours. How much does he earn?\n'
#     '12 x 8 = 96\n'
#     '#### 96\n\n'
# )

# GSM8K_SYSTEM = (
#     'You are a math tutor. Solve the problem step by step, '
#     'then give the final numeric answer on a new line inside \\boxed{}'
# )

# GSM8K_SYSTEM = (
#     'You are a math tutor. Solve the problem step by step, '
#     'then give the final numeric answer inside \\boxed{} on a new line.\n\n'
#     'Example:\nQuestion: Janet has 3 apples. She buys 5 more. How many does she have?\n'
#     'Step 1: 3 + 5 = 8\n'
#     '\\boxed{8}\n\n'
#     'Example:\nQuestion: A store has 20 shirts. They sell 7. How many are left?\n'
#     'Step 1: 20 - 7 = 13\n'
#     '\\boxed{13}\n\n'
# )

def format_gsm8k_raw(example):
    m = re.search(r'####\s*([\d,\.\-]+)', example['answer'])
    numeric = m.group(1).replace(',', '').strip() if m else ''

    # rewrite answer in \boxed{} format instead of native <<>> format
    # strip <<>> annotations and replace #### with \boxed{}
    clean_answer = re.sub(r'<<[^>]*>>', '', example['answer'])  # remove <<calc>>
    clean_answer = re.sub(r'####\s*([\d,\.\-]+)', r'\\boxed{\1}', clean_answer)  # replace ####
    clean_answer = clean_answer.strip()

    return {
        'numeric_answer': numeric,
        'prompt': example['question'].strip(),
        # 'full_answer': example['answer'].strip()
        'full_answer': clean_answer
    }

def gsm8k_apply_chat_template(_base):
    def _apply(example):
        messages = [
            {'role': 'system',    'content': GSM8K_SYSTEM},
            {'role': 'user',      'content': example['prompt']},
            {'role': 'assistant', 'content': example['full_answer']}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {'text': text}
    return _base.map(_apply, num_proc=1, load_from_cache_file=False)

def load_gsm8k_raw(train_samples=None, eval_samples=None, seed=42):
    print('Loading GSM8K ...')
    ds = load_dataset('openai/gsm8k', 'main')
    train = ds['train'].shuffle(seed=seed)
    test  = ds['test'].shuffle(seed=seed)
    if train_samples:
        train = train.select(range(min(train_samples, len(train))))
    if eval_samples:
        test = test.select(range(min(eval_samples, len(test))))
    print(f'  GSM8K — train: {len(train):,} | test: {len(test):,}')
    return train, test



In [ ]:
# ─── Load once, never overwritten ─────────────────────────────────────────────
gsm8k_train_raw, gsm8k_test_raw = load_gsm8k_raw(eval_samples=200)

# model-agnostic formatting — run once
gsm8k_train_base = gsm8k_train_raw.map(format_gsm8k_raw, remove_columns=gsm8k_train_raw.column_names, num_proc=1)
gsm8k_test_base  = gsm8k_test_raw.map(format_gsm8k_raw,  remove_columns=gsm8k_test_raw.column_names,  num_proc=1)
print(f'  GSM8K raw — train: {len(gsm8k_train_base):,} | test: {len(gsm8k_test_base):,}')


Loading GSM8K ...
  GSM8K — train: 7,473 | test: 200


Map (num_proc=1):   0%|          | 0/7473 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/200 [00:00<?, ? examples/s]

  GSM8K raw — train: 7,473 | test: 200


In [ ]:
print(len(set(mmlu_train_raw['subject'])))  # should be 57 for full MMLU
print(len(set(mmlu_train['subject'])))

57
57


## tests

In [ ]:
CHECK_CONFIG = 'late'
adapter_path = os.path.join(OUTPUT_DIR, CHECK_CONFIG, 'adapter')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
check_base  = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True
)
check_model = PeftModel.from_pretrained(check_base, adapter_path)
check_model.eval()

mmlu_metrics = evaluate_mmlu(check_model, tokenizer, mmlu_test)
print(f'Late LoRA MMLU: {mmlu_metrics["mmlu_accuracy"]:.4f} ({mmlu_metrics["mmlu_correct"]}/{mmlu_metrics["mmlu_total"]})')

del check_model, check_base
torch.cuda.empty_cache()

    MMLU examples: 456
    MMLU progress: 200/456 — running acc: 0.4250
    MMLU progress: 400/456 — running acc: 0.4500
Late LoRA MMLU: 0.4474 (204/456)


In [ ]:
# for c in CHOICES:
#     ids = tokenizer.encode(' ' + c, add_special_tokens=False)
#     print(f"' {c}' -> {ids} -> {tokenizer.decode(ids)}")

# mmlu_train = mmlu_apply_chat_template(mmlu_train_base)
# mmlu_test  = mmlu_apply_chat_template(mmlu_test_base)

# # prefix = get_assistant_prefix(tokenizer)
# # print(repr(prefix))

# print(model.config.torch_dtype)
# print(next(model.parameters()).dtype)

# for ex in mmlu_test.select(range(5)):
#     msgs = [{'role': 'user', 'content': ex['prompt']}]
#     prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
#     enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
#     out = model(**enc)
#     last_logits = out.logits[0, -1, :]
#     choice_ids = [tokenizer.encode(' ' + c, add_special_tokens=False)[-1] for c in CHOICES]
#     choice_logits = last_logits[choice_ids]
#     pred = CHOICES[choice_logits.argmax().item()]
#     print(f'gold: {ex["answer_letter"]} | pred: {pred} | logits: {choice_logits.tolist()}')

# tokenizer = load_tokenizer(model_id)
# model = load_base_model(model_id)
# gsm8k_test_sft = gsm8k_apply_chat_template(gsm8k_test_base)
# print(list(gsm8k_test_sft[0].keys()))

# n=range(20)
n=range(200)
# n=[range(0,20), range(20,40), range(40,60)][2] # out of 200 gsm8k_test
# prefix = ['', 'Solution:\n', 'Let me solve step by step.\n\n'][0]

# tk = [256]
tk = [512]

for _ in tk:
    print(f'tk: {_}')
    gsm_metrics = evaluate_gsm8k(
        model, tokenizer, gsm8k_test_sft.select(n), \
        max_new_tokens=_, batch_size=32) # , response_prefix=prefix) # max_new_tokens=64, batch_size=8
    print(f'GSM8K (n={n}): {gsm_metrics["gsm8k_accuracy"]:.4f} ({gsm_metrics["gsm8k_correct"]}/{gsm_metrics["gsm8k_total"]})')

# del model, tokenizer
# cleanup_cuda()

# 256 ####  0.30 0.20 0.25
# 512 ####  0.35 0.25 0.35

In [ ]:
# model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
model_id = 'google/gemma-3-1b-it'
# model_id = 'google/gemma-3-4b-it'
# model_id = 'google/gemma-2-2b-it'
# model_id = 'gghfez/gemma-3-4b-novision'
model = load_base_model(model_id)
tokenizer = load_tokenizer(model_id)

n = range(0,50) # mmlu 50 gsm8k 20
# mmlu_test  = mmlu_apply_chat_template(mmlu_test_base) # full
mmlu_test  = mmlu_apply_chat_template(mmlu_test_base.select(n))
# gsm8k_test  = gsm8k_apply_chat_template(gsm8k_test_base)
# gsm8k_test  = gsm8k_apply_chat_template(gsm8k_test_base.select(n))

t0 = time.time()

mmlu_metrics = evaluate_mmlu(model, tokenizer, mmlu_test, batch_size=8) # mmlu_test
print(f'model: {model_id}\n config: {'base'}\n MMLU acc: {mmlu_metrics["mmlu_accuracy"]} (correct/total: {mmlu_metrics["mmlu_correct"]}/{mmlu_metrics["mmlu_total"]})')

# gsm8k_metrics = evaluate_gsm8k(model, tokenizer, gsm8k_test, batch_size=32) # gsm8k_test bs=64 32, 14b: 16
# print(f'model: {model_id}\n config: {'base'}\n GSM8K acc: {gsm8k_metrics["gsm8k_accuracy"]} (correct/total: {gsm8k_metrics["gsm8k_correct"]}/{gsm8k_metrics["gsm8k_total"]})')

elapsed = time.time() - t0
print(f'Elapsed time: {elapsed:.1f}s')

# del tokenizer
del model, tokenizer
cleanup_cuda()


In [ ]:
# tokenizer = load_tokenizer('meta-llama/Llama-3.2-1B-Instruct')
# mmlu_test = mmlu_apply_chat_template(mmlu_test_base)

sample = mmlu_test[0]
assistant_prefix = get_assistant_prefix(tokenizer)
prompt = sample['text'].split(assistant_prefix)[0] + assistant_prefix + 'Answer:'
print(repr(prompt[-150:]))
print(f'Occurrences of assistant_prefix: {sample["text"].count(assistant_prefix)}')


In [ ]:
print(f'mmlu_test size: {len(mmlu_test)}')
print(f'First example answer: {mmlu_test[0]["answer_letter"]}')
print(f'Last example answer: {mmlu_test[-1]["answer_letter"]}')

## run: base

In [ ]:
## models
# models = [MODEL_ID] ## single model
models = [
    # 'Qwen/Qwen2.5-0.5B',
    'Qwen/Qwen2.5-0.5B-Instruct',
    # 'Qwen/Qwen2.5-1.5B-Instruct',
    # 'Qwen/Qwen2.5-3B-Instruct',
    # 'Qwen/Qwen2.5-7B-Instruct',
    # 'Qwen/Qwen2.5-14B-Instruct',
    # 'Qwen/Qwen3-0.6B',
    # 'Qwen/Qwen3-1.7B',
    # 'Qwen/Qwen3-4B',
    # 'Qwen/Qwen3-4B-Instruct-2507',
    # 'Qwen/Qwen3-8B',
    # 'Qwen/Qwen3-14B',
    # 'meta-llama/Llama-3.2-1B-Instruct',
    # 'meta-llama/Llama-3.2-3B-Instruct',
    # 'meta-llama/Llama-3.1-8B-Instruct',
    # 'google/gemma-3-270m-it',
    # 'google/gemma-3-1b-it',
    # 'google/gemma-3-4b-it',
    # 'gghfez/gemma-3-4b-novision', # bf16 only!
    # 'google/gemma-2-2b-it',
    # 'google/gemma-2-9b-it',
]

## custom setups
# configs = ['all', 'middle', 'late']
# configs = ['late']
configs = ['base']
res = []

def test_config(models, configs):

    for model_id in models:

        for config_name in configs:

            tokenizer = load_tokenizer(model_id)
            check_model = load_base_model(model_id)

            # check_base = load_base_model(model_id)
            # if config_name == 'base':
            #     check_model = check_base
            # else:
            #     check_model = PeftModel.from_pretrained(check_base, os.path.join(OUTPUT_DIR, config_name, 'adapter'))

            check_model.eval()

            t0 = time.time()
            torch.cuda.reset_peak_memory_stats()

            # mmlu_train = mmlu_apply_chat_template(mmlu_train_base)
            mmlu_test  = mmlu_apply_chat_template(mmlu_test_base)
            mmlu_metrics = evaluate_mmlu(check_model, tokenizer, mmlu_test) # mmlu_test
            print(f'model: {model_id}\n config: {config_name}\n MMLU acc: {mmlu_metrics["mmlu_accuracy"]:.4f} (correct/total: {mmlu_metrics["mmlu_correct"]}/{mmlu_metrics["mmlu_total"]})')

            # gsm8k_train = gsm8k_apply_chat_template(gsm8k_train_base)
            gsm8k_test  = gsm8k_apply_chat_template(gsm8k_test_base) #.select(range(n))
            gsm8k_metrics = evaluate_gsm8k(check_model, tokenizer, gsm8k_test, batch_size=32) # gsm8k_test bs=64 32, 14b: 16
            print(f'model: {model_id}\n config: {config_name}\n GSM8K acc: {gsm8k_metrics["gsm8k_accuracy"]:.4f} (correct/total: {gsm8k_metrics["gsm8k_correct"]}/{gsm8k_metrics["gsm8k_total"]})')

            elapsed = time.time() - t0
            print(f'Elapsed time: {elapsed:.2f}s')

            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f'Peak GPU memory during eval: {peak:.2f} GB')

            res.append({
            'model': model_id,
            'config': config_name,
            'mmlu_accuracy': mmlu_metrics["mmlu_accuracy"],
            'gsm8k_accuracy': gsm8k_metrics["gsm8k_accuracy"],
            'peak_vram': peak,
            'time': elapsed,
            })


            ## clear ram
            del check_model, tokenizer
            cleanup_cuda()

test_config(models, configs)


# del check_model, check_base, tokenizer
# cleanup_cuda()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map (num_proc=1): 100%|##########| 456/456 [00:00<?, ? examples/s]

    Assistant prefix: '<|im_end|>\n<|im_start|>assistant\n'
    Choice ids: [('A', 362), ('B', 425), ('C', 356), ('D', 422)]
    Occurrences of assistant_prefix: 1
    MMLU examples: 456
    Batch size: 8
    Tokenizer vocab size: 151643
    Tokenizer padding_side: right
    NaNs in logits: 0
    Logits min/max: -23.62 / 32.19
    MMLU progress: 200/456 — running acc: 0.4550
    MMLU progress: 400/456 — running acc: 0.4725
model: Qwen/Qwen2.5-0.5B-Instruct
 config: base
 MMLU acc: 0.4605 (correct/total: 210/456)


Map (num_proc=1): 100%|##########| 200/200 [00:00<?, ? examples/s]

    Assistant prefix: '<|im_end|>\n<|im_start|>assistant\n'
    GSM8K examples: 200
    Batch size: 32
    Max new tokens: 512
    Tokenizer vocab size: 151643
    Tokenizer padding_side: left
    NaNs in logits: 0
    Logits min/max: -20.95 / 30.88
    Input length: 168 tokens
    Generated tokens: 512
    First batch took 45.9s — estimated total: 4.8 min
    VRAM peak: 1.41 GB / 15.64 GB
    Wrong [1] — Gold: 109 | Pred: 218 | End: "xt{future}} = 208 + 10 \\]\n\\[ A_{\\text{future}} = 218 \\]\n\nTherefore, Allen's age 10 years from now will be 218 years old."
    Wrong [2] — Gold: 89 | Pred: 120 | End: 'ons she got:\n\\[ 60 + 60 = 120 \\text{ buttons} \\]\n\n### Final Answer:\nBy the end, Lorraine has:\n\\[ \\boxed{120} \\]\nbuttons.'
    Wrong [5] — Gold: 25 | Pred: 22 | End: 're, after all the children follow the truck for the entire route, there are \\(\\boxed{22}\\) children following the truck.'
    Wrong [6] — Gold: 452 | Pred: 102 | End: 'nutes she slept:\n- 715 minutes / 7 ho

In [ ]:
pd.DataFrame(res).to_csv('out.csv', sep=';', index=False)
res

[{'model': 'Qwen/Qwen2.5-0.5B-Instruct',
  'config': 'base',
  'mmlu_accuracy': 0.4605263157894737,
  'gsm8k_accuracy': 0.325,
  'peak_vram': 1.421245952,
  'time': 363.1300141811371}]

## 7. run: train

In [ ]:
mmlu_test  = mmlu_apply_chat_template(mmlu_test_base)
gsm8k_test  = gsm8k_apply_chat_template(gsm8k_test_base)
# mmlu_train_sft = mmlu_apply_chat_template(mmlu_train_base)
# print(f'MMLU train: {len(mmlu_train_sft)} examples')
# print(f'Sample text: {repr(mmlu_train_sft[0]["text"][-100:])}')

mmlu_train_sft  = mmlu_apply_chat_template(mmlu_train_base)
gsm8k_train_sft = gsm8k_apply_chat_template(gsm8k_train_base.select(range(min(TRAIN_SAMPLES, len(gsm8k_train_base)))))
combined_train  = concatenate_datasets([mmlu_train_sft, gsm8k_train_sft]).shuffle(seed=42)
print(f'Combined train: {len(combined_train)} examples (MMLU: {len(mmlu_train_sft)} + GSM8K: {len(gsm8k_train_sft)})')


Map (num_proc=1): 100%|##########| 456/456 [00:00<?, ? examples/s]

Map (num_proc=1): 100%|##########| 200/200 [00:00<?, ? examples/s]

Map (num_proc=1): 100%|##########| 1816/1816 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/7473 [00:00<?, ? examples/s]

Combined train: 9289 examples (MMLU: 1816 + GSM8K: 7473)


In [ ]:

cleanup_cuda()
res = run_experiment('late', LAYER_CONFIGS['late'])
# res = run_experiment('mid', LAYER_CONFIGS['mid'])
# res = run_experiment('early', LAYER_CONFIGS['early'])
# res = run_experiment('all', LAYER_CONFIGS['all'])
# res = run_experiment('rand', LAYER_CONFIGS['rand'])


  Freed ~0.00 GB | Allocated: 0.02 / 15.64 GB | Reserved: 0.47 GB

EXPERIMENT: LATE  (8 layers)

[1/4] Loading base model ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Response template: '<|im_end|>\n<|im_start|>assistant\n'
Response template ids: [151645, 198, 151644, 77091, 198]
[2/4] Applying LoRA ...
[late] Trainable params: 1,466,368 / 495,499,136 (0.30%)
[3/4] Fine-tuning ...


Map:   0%|          | 0/9289 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,0.885649
40,0.727570
60,0.666284
80,0.624787
100,0.663994
120,0.618134
140,0.628450
160,0.629395
180,0.630284
200,0.637498


Train time: 694.04s
Peak VRAM train: 10.71 GB
  Adapter saved to /content/drive/MyDrive/LoRA/Data/late/adapter
[4/4] Evaluating ...
LoRA keys count: 56
First LoRA key: base_model.model.model.layers.16.self_attn.q_proj.lora_A.default.weight
First LoRA sum: -2.200777
Last LoRA key: base_model.model.model.layers.23.mlp.down_proj.lora_A.default.weight
Last LoRA sum: -1.259672
Model training mode: False
Active adapters: ['default']
    Assistant prefix: '<|im_end|>\n<|im_start|>assistant\n'
    Choice ids: [('A', 362), ('B', 425), ('C', 356), ('D', 422)]
    Occurrences of assistant_prefix: 1
    MMLU examples: 456
    Batch size: 8
    Tokenizer vocab size: 151643
    Tokenizer padding_side: right
    NaNs in logits: 0
    Logits min/max: -20.58 / 31.14
    MMLU progress: 200/456 — running acc: 0.4250


In [ ]:
# cleanup_cuda()
res
# all_results

{'config': 'early',
 'n_layers': 8,
 'layer_range': '0–7',
 'train_loss': 0.6168252381984358,
 'trainable_params': 1466368,
 'mmlu_accuracy': 0.4407894736842105,
 'mmlu_correct': 201,
 'mmlu_total': 456,
 'gsm8k_accuracy': 0.25,
 'gsm8k_correct': 50,
 'gsm8k_total': 200,
 'vram_train': 10.704441344,
 'time_train': 274.8727400302887,
 'vram_eval': 0.833468928,
 'time_eval': 273.4921147823334}

In [ ]:
print(repr(gsm8k_train_sft[0]['text'][-200:]))

# what format does evaluate_gsm8k use for prompts?
print(repr(gsm8k_test[0]['text'].split('<|im_start|>assistant')[0][-100:]))

'yle found.  How many seashells did Leigh have?<|im_end|>\n<|im_start|>assistant\nMimi has 2 x 12 = 24 sea shells.\nKyle has 24 x 2 = 48 sea shells.\nLeigh has 48 / 3 = 16 sea shells.\n\\boxed{16}<|im_end|>\n'
"he ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.<|im_end|>\n"


In [ ]:
import safetensors.torch as st
for config in ['early', 'mid', 'late']:
    w = st.load_file(os.path.join(OUTPUT_DIR, config, 'adapter', 'adapter_model.safetensors'))
    first_key = list(w.keys())[0]
    print(f'{config}: first key = {first_key} | sum = {w[first_key].sum().item():.4f}')


early: first key = base_model.model.model.layers.0.mlp.down_proj.lora_A.weight | sum = -2.6689
mid: first key = base_model.model.model.layers.10.mlp.down_proj.lora_A.weight | sum = 0.3248
late: first key = base_model.model.model.layers.16.mlp.down_proj.lora_A.weight | sum = -1.9454


In [ ]:
# ─── Run experiments sequentially ─────────────────────────────────────────────
# Each run unloads the model after finishing to stay within T4 VRAM (15 GB).

all_results = []

for config_name, layer_indices in LAYER_CONFIGS.items():
    result = run_experiment(config_name, layer_indices)
    all_results.append(result)
    print(f'\nFinished [{config_name}] in {result["time_train"]/60:.1f} min\n')

print('\nAll experiments complete!')


EXPERIMENT: ALL  (26 layers)

[1/4] Loading base model ...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Response template: '<end_of_turn>\n<start_of_turn>model\n'
Response template ids: [107, 108, 106, 2516, 108]
[2/4] Applying LoRA ...


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[all] Trainable params: 10,383,360 / 2,624,725,248 (0.40%)
[3/4] Fine-tuning ...


Map:   0%|          | 0/1816 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
20,0.000000
40,0.000000


KeyboardInterrupt: 

## 8. Results & Analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.DataFrame(all_results)

display_cols = [
    'config', 'n_layers', 'layer_range', 'trainable_params',
    'mmlu_accuracy', 'gsm8k_accuracy', 'train_loss', 'train_time_s'
]
df_display = df[display_cols].copy()
df_display['trainable_params'] = df_display['trainable_params'].apply(lambda x: f'{x:,}')
df_display['mmlu_accuracy']    = df_display['mmlu_accuracy'].apply(lambda x: f'{x:.4f}')
df_display['gsm8k_accuracy']   = df_display['gsm8k_accuracy'].apply(lambda x: f'{x:.4f}')
df_display['train_loss']       = df_display['train_loss'].apply(lambda x: f'{x:.4f}')
df_display['train_time_s']     = df_display['train_time_s'].apply(lambda x: f'{x/60:.1f} min')

print('\n=== RESULTS SUMMARY ===')
display(df_display.set_index('config'))

In [ ]:
# ─── Grouped bar chart ────────────────────────────────────────────────────────
configs = [r['config'] for r in all_results]
mmlu_accs = [r['mmlu_accuracy'] for r in all_results]
gsm_accs  = [r['gsm8k_accuracy'] for r in all_results]

x = np.arange(len(configs))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Qwen2.5-0.5B-Instruct — LoRA Layer Ablation', fontsize=14, fontweight='bold')

# Left: accuracy comparison
ax = axes[0]
bars1 = ax.bar(x - width/2, mmlu_accs, width, label='MMLU', color='#4A90D9', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + width/2, gsm_accs,  width, label='GSM8K', color='#E87040', edgecolor='white', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f'{c}\n({LAYER_CONFIGS[c][0]}–{LAYER_CONFIGS[c][-1]})' for c in configs])
ax.set_ylabel('Accuracy')
ax.set_title('Benchmark Accuracy by Layer Config')
ax.legend()
ax.set_ylim(0, min(1.0, max(mmlu_accs + gsm_accs) * 1.3))
for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

# Right: trainable params vs accuracy
ax2 = axes[1]
params = [r['trainable_params'] / 1e6 for r in all_results]
colors = ['#4A90D9', '#E87040', '#50C878']
for i, (p, ma, ga, cfg, col) in enumerate(zip(params, mmlu_accs, gsm_accs, configs, colors)):
    ax2.scatter(p, ma, s=120, c=col, marker='o', label=f'{cfg} (MMLU)', zorder=5)
    ax2.scatter(p, ga, s=120, c=col, marker='s', label=f'{cfg} (GSM8K)', zorder=5)
    ax2.annotate(cfg, (p, max(ma, ga)), textcoords='offset points', xytext=(5, 5), fontsize=8)

ax2.set_xlabel('Trainable Parameters (M)')
ax2.set_ylabel('Accuracy')
ax2.set_title('Efficiency: Params vs Accuracy')
circle_patch = mpatches.Patch(color='gray', label='○ = MMLU  □ = GSM8K')
ax2.legend(handles=[circle_patch], loc='lower right', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'layer_ablation_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 9. Inference with a Saved Adapter

In [ ]:
# ─── Load a saved adapter for inference ───────────────────────────────────────
# Change LOAD_CONFIG to 'middle' or 'late' to compare
LOAD_CONFIG = 'all'
adapter_path = os.path.join(OUTPUT_DIR, LOAD_CONFIG, 'adapter')

from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

inf_base  = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True
)
inf_model = PeftModel.from_pretrained(inf_base, adapter_path)
inf_model.eval()


def ask(question: str, system: str = '') -> str:
    """Generate a response with the loaded adapter."""
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': question})
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = inf_model.generate(
            **enc, max_new_tokens=200, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)


# ─── Example: MMLU question ───────────────────────────────────────────────────
mmlu_q = (
    'Answer with only the letter.\n\n'
    'Question: What is the derivative of sin(x)?\n'
    'A. -cos(x)\nB. cos(x)\nC. -sin(x)\nD. tan(x)\n\nAnswer:'
)
print('MMLU example:')
print(ask(mmlu_q))

# ─── Example: GSM8K question ──────────────────────────────────────────────────
gsm_q = 'Janet has 3 apples. She buys 5 more and gives 2 to her friend. How many apples does she have?'
print('\nGSM8K example:')
print(ask(gsm_q, system=GSM8K_SYSTEM))

## 10. Save Full Results CSV

In [ ]:
csv_path = os.path.join(OUTPUT_DIR, 'layer_ablation_summary.csv')
df.to_csv(csv_path, index=False)
print(f'Results saved to: {csv_path}')

# Also print a markdown table for easy copy-paste
print('\n### Markdown Summary')
print(df[['config', 'n_layers', 'layer_range', 'mmlu_accuracy', 'gsm8k_accuracy', 'train_loss']].to_markdown(index=False))

## Notes & Tips

### Memory
- Each run loads the 0.5B model in 4-bit (QLoRA). Peak VRAM is ~4–6 GB, well within T4 limits.
- If you hit OOM, reduce `TRAIN_SAMPLES`, `BATCH_SIZE`, or `MAX_SEQ_LEN`.

### Layer config tuning
| Config | Blocks | Notes |
|--------|--------|-------|
| `all` | 0–23 | Most trainable params, slowest |
| `middle` | 7–21 | Mid-level reasoning, language |
| `late` | 14–23 | Output-side features, factual recall |

### Longer training
Set `TRAIN_EPOCHS = 3` and `TRAIN_SAMPLES = 5000` for production-quality results.  
Expected wall-clock on T4: ~40 min per experiment.

### Full MMLU benchmark
Set `EVAL_SAMPLES = None` and include all 57 MMLU subjects for a publishable number.

### Merging adapters
```python
merged = inf_model.merge_and_unload()
merged.save_pretrained('merged_model')
```

## msc

In [ ]:
### gemma-3-4b-it multimodal to language

# So the complete working recipe for gemma-3-4b-it text-only is:
# 1 Load full multimodal model in 4-bit
# 2 Dequantize weights using bnb.functional.dequantize_4bit
# 3 Remap keys from model.language_model.* → model.*
# 4 Tie lm_head.weight to embed_tokens.weight
# 5 Load into Gemma3ForCausalLM with text_config
# 6 Convert to bfloat16 (not float16) before moving to GPU
# The float16 → NaN issue is the key insight — Gemma 3 uses bfloat16 internally and float16's smaller dynamic range causes overflow.


from transformers import Gemma3ForConditionalGeneration, BitsAndBytesConfig
import bitsandbytes as bnb
import gc

model_id = 'google/gemma-3-4b-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f'Before loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

full = Gemma3ForConditionalGeneration.from_pretrained(
    'google/gemma-3-4b-it',
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
print(f'After loading full: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

# state_dict = {}
# for k, v in full.state_dict().items():
#     if k.startswith('model.language_model.'):
#         new_k = k.removeprefix('model.language_model.')
#         # filter out bitsandbytes quantization auxiliary keys
#         if any(x in new_k for x in ['absmax', 'quant_map', 'quant_state', 'quant_type', 'nested']):
#             continue
#         if new_k == 'lm_head.weight':
#             state_dict['lm_head.weight'] = v.to('cpu')
#         else:
#             state_dict['model.' + new_k] = v.to('cpu')

state_dict = {}
for name, module in full.named_modules():
    if not name.startswith('model.language_model.'):
        continue
    new_name = 'model.' + name.removeprefix('model.language_model.')
    if isinstance(module, bnb.nn.Linear4bit):
        # dequantize the 4-bit weight back to float16
        weight = bnb.functional.dequantize_4bit(
            module.weight.data,
            module.weight.quant_state
        ).to(torch.float16)
        state_dict[new_name + '.weight'] = weight.to('cpu')
        if module.bias is not None:
            state_dict[new_name + '.bias'] = module.bias.data.to('cpu')
    elif hasattr(module, 'weight') and module.weight is not None:
        if not any(isinstance(module, t) for t in [bnb.nn.Linear4bit]):
            state_dict[new_name + '.weight'] = module.weight.data.to(torch.float16).to('cpu')
            if hasattr(module, 'bias') and module.bias is not None:
                state_dict[new_name + '.bias'] = module.bias.data.to(torch.float16).to('cpu')

# lm_head is tied to embed_tokens in Gemma
if 'model.embed_tokens.weight' in state_dict:
    state_dict['lm_head.weight'] = state_dict['model.embed_tokens.weight']
    print('lm_head.weight tied to embed_tokens.weight')


print(f'State dict keys: {len(state_dict)}')
print('First 5 remapped keys:')
for k in list(state_dict.keys())[:5]:
    print(f'  {k}')
print(f'embed_tokens.weight in state_dict: {"embed_tokens.weight" in state_dict}')

del full
gc.collect()
torch.cuda.empty_cache()
print(f'After del full: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

text_config = AutoConfig.from_pretrained(model_id).text_config

# model = Gemma3ForCausalLM(text_config)
with torch.device('cpu'):
    model = Gemma3ForCausalLM(text_config).to(torch.float16) # force CPU allocation

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f'Missing: {len(missing)} | Unexpected: {len(unexpected)}')
print('First 5 missing:')
for k in missing[:5]:
    print(f'  {k}')

del state_dict
torch.cuda.empty_cache()
gc.collect()

print(f'VRAM before move: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

# model = model.to(DEVICE)
model = model.to(torch.bfloat16).to(DEVICE)

print(f'VRAM after move: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

model.config.use_cache = False
model.enable_input_require_grads()


Before loading: 0.01 GB


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

After loading full: 3.42 GB
lm_head.weight tied to embed_tokens.weight
State dict keys: 445
First 5 remapped keys:
  model.embed_tokens.weight
  model.layers.0.self_attn.q_proj.weight
  model.layers.0.self_attn.k_proj.weight
  model.layers.0.self_attn.v_proj.weight
  model.layers.0.self_attn.o_proj.weight
embed_tokens.weight in state_dict: False
After del full: 1.40 GB
Missing: 0 | Unexpected: 0
First 5 missing:
VRAM before move: 1.40 GB
VRAM after move: 9.16 GB


In [ ]:
### single vs batched accuracy debug

model.eval()
choice_ids = [tokenizer.encode(' ' + c, add_special_tokens=False)[-1] for c in CHOICES]
examples = list(mmlu_test)[:8]

prompts, golds = [], []
for ex in examples:
    text = ex['text']
    prompt = text.split('<start_of_turn>model\n')[0] + '<start_of_turn>model\nAnswer:'
    prompts.append(prompt)
    golds.append(ex['answer_letter'])

# Batched
enc = tokenizer(prompts, return_tensors='pt', padding=True,
                truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
with torch.no_grad():
    out = model(**enc)
attention_mask = enc['attention_mask']
last_token_pos = attention_mask.sum(dim=1) - 1
last_logits = out.logits[torch.arange(out.logits.shape[0], device=out.logits.device), last_token_pos]
choice_logits = last_logits[:, choice_ids]
preds_batched = [CHOICES[j] for j in choice_logits.argmax(dim=-1).tolist()]

# Single
preds_single = []
for prompt in prompts:
    enc_s = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    with torch.no_grad():
        out_s = model(**enc_s)
    last_logits_s = out_s.logits[0, -1, :]
    preds_single.append(CHOICES[last_logits_s[choice_ids].argmax().item()])
    print(f'  {gold}  |   {pb}    |   {ps}   | batched_ok={pb==gold} single_ok={ps==gold}')

print('Gold | Batched | Single | Match')
for gold, pb, ps in zip(golds, preds_batched, preds_single):

Gold | Batched | Single | Match
  C  |   A    |   C   | batched_ok=False single_ok=True
  C  |   D    |   C   | batched_ok=False single_ok=True
  D  |   A    |   D   | batched_ok=False single_ok=True
  B  |   A    |   A   | batched_ok=False single_ok=False
  B  |   A    |   C   | batched_ok=False single_ok=False
  B  |   A    |   B   | batched_ok=False single_ok=True
  C  |   D    |   C   | batched_ok=False single_ok=True
  D  |   A    |   D   | batched_ok=False single_ok=True
